# NFL Play by Play 2009–2017 (v4) — Pré-processamento e armazenamento

**Atividade individual | Python, Pandas e Parquet**

## Objetivo
Preparar um conjunto de dados consistente, confiável e otimizado para armazenamento, documentando diagnóstico, tipificação, valores ausentes, limpeza, exportação e validação.

### Como este notebook foi montado
Este notebook reúne e reorganiza os trechos relevantes dos dois notebooks fornecidos: o gabarito (`Pre_processamento_dados_Gabarito`) e a segunda parte (`Pre_processamento_dados_2`). Foram mantidos os conceitos e os códigos-base de inspeção de tipos mistos, diagnóstico de `object`, análise de missing por faixa, tipificação e gravação em Parquet. Os trechos foram adaptados quando necessário para evitar caminhos locais fixos, erros quando uma coluna não existe e alterações destrutivas sem justificativa.


## 1. Bibliotecas e configurações

O gabarito utiliza `os`, `numpy` e `pandas`; a segunda parte também utiliza `matplotlib` para visualização. Para exportação e leitura Parquet, o ambiente precisa ter `pyarrow` instalado.

In [ ]:
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Se necessário, informe aqui o caminho completo para o CSV original.
# Exemplo no Kaggle: /kaggle/input/nfl-play-by-play-2009-2017/NFL Play by Play 2009-2017 (v4).csv
CSV_PATH = None
NOME_CSV = "NFL Play by Play 2009-2017 (v4).csv"
NOME_PARQUET = "NFL_Play_by_Play_2009-2017_tratado.parquet"

## 2. Carregamento do dataset

No gabarito, o arquivo é obtido com `kagglehub.dataset_download("maxhorowitz/nflplaybyplay2009to2016")` e depois localizado com `os.walk`. Aqui mantemos essa abordagem como alternativa, mas primeiro procuramos um arquivo já adicionado ao ambiente. Isso também evita depender de um caminho de computador específico.

In [ ]:
def localizar_csv():
    """Localiza o CSV configurado, no Kaggle ou no diretório de trabalho."""
    if CSV_PATH:
        caminho = Path(CSV_PATH).expanduser()
        if caminho.is_file():
            return caminho
        raise FileNotFoundError(f"CSV_PATH foi informado, mas o arquivo não existe: {caminho}")

    # Caminhos usuais do Kaggle e do ambiente local.
    diretorios = [Path("/kaggle/input"), Path.cwd(), Path("/mnt/data")]
    for diretorio in diretorios:
        if diretorio.exists():
            encontrados = list(diretorio.rglob(NOME_CSV))
            if encontrados:
                return encontrados[0]

    # Alternativa inspirada no gabarito: baixar pelo KaggleHub, se disponível.
    try:
        import kagglehub
        raiz = Path(kagglehub.dataset_download("maxhorowitz/nflplaybyplay2009to2016"))
        encontrados = list(raiz.rglob(NOME_CSV))
        if encontrados:
            return encontrados[0]
    except Exception as erro:
        warnings.warn(f"Não foi possível localizar/baixar pelo KaggleHub: {erro}")

    raise FileNotFoundError(
        f"Não encontrei '{NOME_CSV}'. Adicione o dataset ao Kaggle ou defina CSV_PATH com o caminho do CSV."
    )

csv_path = localizar_csv()
print("Arquivo de origem:", csv_path)
print("Tamanho do CSV (bytes):", csv_path.stat().st_size)

# low_memory=False reduz avisos de inferência por blocos; as colunas mistas ainda serão inspecionadas.
df_original = pd.read_csv(csv_path, low_memory=False)
df = df_original.copy()
print("Dataset carregado com sucesso.")

## 3. Diagnóstico estrutural

Esta etapa mantém as verificações do gabarito (`df.info`, dimensões e tipos) e da segunda parte (`head`, `shape` e contagem de missing). O objetivo é registrar a estrutura antes de modificar qualquer dado.

In [ ]:
display(df.head())
print("Dimensões (linhas, colunas):", df.shape)
df.info(verbose=True, show_counts=True)

resumo_tipos = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nao_ausentes": df.notna().sum(),
    "ausentes": df.isna().sum(),
    "percentual_ausente_%": (df.isna().mean() * 100).round(2),
    "valores_unicos_sem_na": df.nunique(dropna=True)
}).sort_values("percentual_ausente_%", ascending=False)
display(resumo_tipos)
print("Quantidade de colunas por tipo técnico:")
display(df.dtypes.astype(str).value_counts().rename_axis("tipo").to_frame("quantidade"))

## 4. Inspeção de tipos mistos e colunas `object`

No gabarito, as colunas de índices 25 e 51 foram investigadas por causa do `DtypeWarning`. Os índices podem variar conforme a versão do arquivo; por isso, o código verifica se existem antes de inspecionar. Em seguida, listamos as colunas `object` e algumas amostras para decidir a tipagem com base no significado dos atributos.

In [ ]:
# Trecho baseado no gabarito: investigar as colunas 25 e 51 se existirem.
indices_suspeitos = [i for i in [25, 51] if i < len(df.columns)]
problem_cols = df.columns[indices_suspeitos].tolist()
print("Colunas investigadas por posição:", problem_cols)

for coluna in problem_cols:
    print(f"\n--- {coluna}: tipos Python encontrados ---")
    display(df[coluna].map(lambda valor: type(valor).__name__).value_counts(dropna=False))
    print("Valores únicos (até 30):")
    display(pd.Series(df[coluna].dropna().unique()).head(30))
    print("Ausentes:", int(df[coluna].isna().sum()))

object_cols = df.select_dtypes(include="object").columns.tolist()
print("Total de variáveis do tipo object:", len(object_cols))
print(object_cols)


data_dict_object = pd.DataFrame({
    "variavel": object_cols,
    "dtype_atual": [str(df[col].dtype) for col in object_cols],
    "valores_unicos": [df[col].nunique(dropna=True) for col in object_cols],
    "amostras": [", ".join(df[col].dropna().astype(str).unique()[:8]) for col in object_cols]
})
display(data_dict_object)

## 5. Análise de valores ausentes

O gabarito destaca que `isna()` detecta ausências formais (`NaN`, `None`, `pd.NA`, `NaT`), mas não identifica sozinho textos como `"NA"`, `"null"` ou strings vazias. A segunda parte agrupa as colunas por faixas percentuais de ausência. Reproduzimos essas análises antes de decidir o tratamento.

**Decisão:** os percentuais são instrumentos de diagnóstico, não uma regra automática para apagar colunas. Em dados de jogadas, uma ausência pode significar que o atributo não se aplica àquela jogada (por exemplo, dados de passe em uma corrida). Portanto, não vamos remover colunas apenas por terem muitos valores ausentes.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)
tabela_missing = pd.DataFrame({
    "ausentes": missing,
    "percentual_%": missing_percent.round(2)
})
display(tabela_missing)

# Verificação de tokens textuais comuns em colunas de texto.
missing_tokens = {"", "na", "n/a", "null", "none", "nan", "unknown"}
tokens_encontrados = {}
for col in df.select_dtypes(include=["object", "string"]).columns:
    valores = df[col].astype("string").str.strip().str.lower()
    qtd = int(valores.isin(missing_tokens).sum())
    if qtd:
        tokens_encontrados[col] = qtd
print("Tokens textuais potencialmente ausentes por coluna:")
display(pd.Series(tokens_encontrados, name="ocorrencias").sort_values(ascending=False))

# Faixas percentuais usadas no segundo notebook.
missing_50 = missing_percent[missing_percent >= 50].index
missing_30_50 = missing_percent[(missing_percent >= 30) & (missing_percent < 50)].index
missing_20_30 = missing_percent[(missing_percent >= 20) & (missing_percent < 30)].index
missing_10_20 = missing_percent[(missing_percent >= 10) & (missing_percent < 20)].index
missing_5_10 = missing_percent[(missing_percent >= 5) & (missing_percent < 10)].index
missing_5 = missing_percent[missing_percent < 5].index

for nome, colunas in [
    ("50% ou mais", missing_50), ("30% a menos de 50%", missing_30_50),
    ("20% a menos de 30%", missing_20_30), ("10% a menos de 20%", missing_10_20),
    ("5% a menos de 10%", missing_5_10), ("menos de 5%", missing_5)
]:
    print(f"{nome}: {len(colunas)} coluna(s)")

### 5.1 Inspeção de valores sentinela

O gabarito sugere examinar valores como `-1` e `0`, mas também alerta que é necessário conhecer o domínio. Neste dataset, `0` pode ser um valor válido (por exemplo, pontuação ou jardas), e `-1` pode aparecer como código especial. Assim, o código abaixo apenas contabiliza ocorrências; não as converte automaticamente em missing.

In [ ]:
sentinelas = {}
for col in df.columns:
    try:
        qtd = int(df[col].isin([-1, 0, "UNKNOWN", "unknown"]).sum())
        if qtd:
            sentinelas[col] = qtd
    except (TypeError, ValueError):
        continue
print("Ocorrências de valores candidatos a sentinela:")
display(pd.Series(sentinelas, name="ocorrencias").sort_values(ascending=False))

## 6. Tipificação dos atributos

O gabarito diferencia tipo técnico e tipo semântico: por exemplo, nomes de atletas são texto, nomes de times e resultados de jogada são categorias, datas são temporais e identificadores não devem ser tratados como medidas numéricas. A tipagem abaixo reaproveita as listas do gabarito, mas converte somente colunas que realmente existem no arquivo.

**Decisões de tipagem:**
- Times, resultados e classificações de domínio limitado: `category`.
- Nomes, descrições e identificadores textuais: `string`.
- `Date`: data e hora, com `errors="coerce"` para permitir a contagem de datas inválidas.
- `time`: duração, usando `to_timedelta`; valores que não puderem ser interpretados ficam ausentes e são contabilizados.
- Variáveis numéricas permanecem numéricas quando sua semântica é quantitativa.

In [ ]:
def converter_se_existirem(dataframe, colunas, dtype):
    """Converte apenas as colunas presentes, evitando KeyError entre versões do dataset."""
    presentes = [col for col in colunas if col in dataframe.columns]
    if presentes:
        dataframe[presentes] = dataframe[presentes].astype(dtype)
    ausentes = [col for col in colunas if col not in dataframe.columns]
    if ausentes:
        print(f"Colunas ignoradas por não existirem no arquivo ({dtype}): {ausentes}")

# Listas originais do gabarito, com verificação de existência.
acronyms = ['HomeTeam', 'Timeout_Team', 'AwayTeam', 'posteam', 'DefensiveTeam',
             'RecFumbTeam', 'PenalizedTeam', 'SideofField']
names = ['Rusher', 'Tackler1', 'Receiver', 'Returner', 'PenalizedPlayer',
         'Tackler2', 'RecFumbPlayer', 'Interceptor', 'Passer']
id_player = ['Rusher_ID', 'Receiver_ID', 'Passer_ID']
unique = ['PassLength', 'ExPointResult', 'PuntResult', 'FieldGoalResult', 'PenaltyType',
          'PassOutcome', 'PlayType', 'TwoPointConv', 'ReturnResult', 'ChalReplayResult',
          'RunGap', 'RunLocation', 'PassLocation']

converter_se_existirem(df, acronyms, 'category')
converter_se_existirem(df, names, 'string')
converter_se_existirem(df, id_player, 'string')
converter_se_existirem(df, unique, 'category')

# Alguns identificadores adicionais são textuais, não medidas numéricas.
converter_se_existirem(df, ['GameID'], 'string')

# Data do jogo: preserva valores não interpretáveis como NaT e registra quantos foram afetados.
if 'Date' in df.columns:
    ausentes_antes = int(df['Date'].isna().sum())
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    invalidas_novas = int(df['Date'].isna().sum()) - ausentes_antes
    print("Datas inválidas adicionais convertidas para NaT:", max(invalidas_novas, 0))

# Horário da jogada: no gabarito, a coluna time é interpretada como duração em minutos/segundos.
if 'time' in df.columns:
    ausentes_antes = int(df['time'].isna().sum())
    texto_time = df['time'].astype('string').str.strip()
    # Aceita valores no formato MM:SS; já valores com HH:MM:SS também são aceitos.
    texto_time = texto_time.where(texto_time.isna() | texto_time.str.count(':').eq(2), '00:' + texto_time)
    df['time'] = pd.to_timedelta(texto_time, errors='coerce')
    invalidas_novas = int(df['time'].isna().sum()) - ausentes_antes
    print("Horários inválidos adicionais convertidos para NaT:", max(invalidas_novas, 0))

if 'desc' in df.columns:
    df['desc'] = df['desc'].astype('string')

print("Tipos após tipificação:")
display(df.dtypes.astype(str).value_counts().rename_axis("tipo").to_frame("quantidade"))

## 7. Tratamento justificado de valores ausentes

Na segunda parte, a variável `down` é analisada porque valores ausentes podem representar jogadas nas quais não existe uma tentativa ofensiva (como kickoff ou field goal). O código original propõe preencher `down` com `0` e converter para categoria. Essa decisão só é aplicada se a coluna existir e deve ser interpretada como uma categoria explícita **“sem down aplicável”**, não como uma medição numérica.

Para as demais colunas, não faremos imputação indiscriminada por média, zero ou remoção de linhas. WPA, por exemplo, representa o impacto de uma jogada na probabilidade de vitória; estimá-lo exige um modelo e uma justificativa analítica próprios. Como o objetivo principal aqui é preparar e armazenar os dados preservando a informação original, valores ausentes são mantidos quando não há regra de domínio suficientemente segura.

In [ ]:
if 'down' in df.columns:
    ausentes_down_antes = int(df['down'].isna().sum())
    # Atribuir 0 significa explicitamente "não se aplica", conforme a hipótese do notebook de aula.
    df['down'] = df['down'].fillna(0).astype('category')
    print("Valores ausentes em down antes:", ausentes_down_antes)
    print("Valores ausentes em down depois:", int(df['down'].isna().sum()))
    print("Frequências de down, incluindo a categoria 0:")
    display(df['down'].value_counts(dropna=False).sort_index())
else:
    print("A coluna 'down' não existe nesta versão; nenhuma imputação foi aplicada.")

# Tabela de ausências depois das decisões documentadas.
missing_depois_tratamento = pd.DataFrame({
    'ausentes': df.isna().sum(),
    'percentual_%': (df.isna().mean() * 100).round(2)
}).sort_values('percentual_%', ascending=False)
display(missing_depois_tratamento)

## 8. Limpeza de duplicidades e revisão de qualidade

Removemos apenas linhas completamente idênticas. Não removemos linhas porque compartilham `GameID`: um jogo contém muitas jogadas, portanto a repetição do identificador é esperada. Também não excluímos colunas com muitos missing automaticamente, pois a ausência pode ter significado no contexto da jogada.

In [ ]:
linhas_antes_dedup = len(df)
duplicadas_exatas = int(df.duplicated(keep='first').sum())
df = df.drop_duplicates(keep='first').reset_index(drop=True)
linhas_depois_dedup = len(df)

print("Linhas antes da remoção de duplicatas exatas:", linhas_antes_dedup)
print("Duplicatas exatas encontradas:", duplicadas_exatas)
print("Linhas depois:", linhas_depois_dedup)

# Relatório simples de qualidade: infinitos em colunas numéricas e colunas constantes.
colunas_numericas = df.select_dtypes(include=[np.number]).columns
infinitos = {col: int(np.isinf(df[col].to_numpy(dtype=float, na_value=np.nan)).sum())
             for col in colunas_numericas
             if np.isinf(df[col].to_numpy(dtype=float, na_value=np.nan)).any()}
constantes = [col for col in df.columns if df[col].nunique(dropna=False) <= 1]
print("Valores infinitos em colunas numéricas:")
display(pd.Series(infinitos, name='quantidade'))
print("Colunas constantes (apenas relatório; não removidas automaticamente):", constantes)

## 9. Armazenamento em Parquet e comparação de tamanho

O gabarito exporta o DataFrame com `to_parquet(..., index=False)`. Mantemos essa abordagem, adicionando compressão `zstd` para tentar reduzir o tamanho do arquivo. O arquivo Parquet preserva tipos de coluna melhor do que CSV e é adequado para leitura analítica. A biblioteca `pyarrow` precisa estar instalada.

In [ ]:
parquet_path = Path.cwd() / NOME_PARQUET
try:
    df.to_parquet(parquet_path, index=False, engine='pyarrow', compression='zstd')
except ImportError as erro:
    raise ImportError("Para exportar Parquet, instale pyarrow: pip install pyarrow") from erro

csv_bytes = csv_path.stat().st_size
parquet_bytes = parquet_path.stat().st_size
reducao_pct = (1 - parquet_bytes / csv_bytes) * 100 if csv_bytes else np.nan

print("CSV original:", f"{csv_bytes:,} bytes ({csv_bytes / (1024**2):.2f} MiB)")
print("Parquet tratado:", f"{parquet_bytes:,} bytes ({parquet_bytes / (1024**2):.2f} MiB)")
print("Redução percentual do tamanho:", f"{reducao_pct:.2f}%" if pd.notna(reducao_pct) else "indisponível")
print("Arquivo Parquet salvo em:", parquet_path.resolve())

## 10. Validação após armazenamento

Reabrimos o Parquet e verificamos dimensões, nomes e ordem das colunas, valores e tipos. Para comparação de valores, `assert_frame_equal` é usado com tolerância a diferenças de representação de tipos que podem ocorrer entre bibliotecas. Se uma diferença for encontrada, ela deve ser investigada antes da entrega.

In [ ]:
from pandas.testing import assert_frame_equal

df_lido = pd.read_parquet(parquet_path, engine='pyarrow')
mesma_forma = df.shape == df_lido.shape
mesmas_colunas = df.columns.tolist() == df_lido.columns.tolist()

try:
    assert_frame_equal(df, df_lido, check_dtype=True, check_categorical=True, check_like=False)
    mesmos_valores_e_tipos = True
    detalhe_validacao = "DataFrame preservado: valores, ordem das colunas e tipos conferem."
except AssertionError as erro:
    mesmos_valores_e_tipos = False
    detalhe_validacao = str(erro)[:1500]

print("Mesma dimensão:", mesma_forma)
print("Mesmas colunas e ordem:", mesmas_colunas)
print("Mesmos valores e tipos:", mesmos_valores_e_tipos)
print(detalhe_validacao)

if not (mesma_forma and mesmas_colunas and mesmos_valores_e_tipos):
    warnings.warn("A validação encontrou diferenças; revise a mensagem acima antes de entregar.")

## 11. Análise final e resumo das transformações

A célula seguinte monta um resumo com valores reais da execução. A interpretação final deve mencionar os números apresentados, sem antecipar resultados que dependem do arquivo de entrada.

In [ ]:
resumo = pd.Series({
    'arquivo_origem': str(csv_path),
    'arquivo_saida': str(parquet_path.resolve()),
    'linhas_originais': int(len(df_original)),
    'linhas_finais': int(len(df)),
    'colunas_originais': int(df_original.shape[1]),
    'colunas_finais': int(df.shape[1]),
    'duplicatas_exatas_removidas': int(duplicadas_exatas),
    'ausentes_formais_antes': int(df_original.isna().sum().sum()),
    'ausentes_formais_depois': int(df.isna().sum().sum()),
    'tamanho_csv_bytes': int(csv_bytes),
    'tamanho_parquet_bytes': int(parquet_bytes),
    'reducao_tamanho_percentual': round(float(reducao_pct), 2) if pd.notna(reducao_pct) else np.nan,
    'validacao_dimensao': bool(mesma_forma),
    'validacao_colunas': bool(mesmas_colunas),
    'validacao_valores_tipos': bool(mesmos_valores_e_tipos)
})
display(resumo.to_frame('resultado'))

### Conclusão

Ao executar todas as células, registre na conclusão se o arquivo Parquet ficou menor que o CSV, quantas duplicatas exatas foram removidas e se a validação confirmou a preservação dos dados. As decisões principais foram: tipar atributos conforme seu significado; quantificar ausências antes de tratá-las; representar `down` ausente como `0` apenas sob a interpretação de “não aplicável”; não imputar outras variáveis automaticamente; não apagar colunas somente pelo percentual de missing; e remover somente duplicatas de linha exatamente iguais.

### Observação sobre WPA e imputação múltipla
A segunda parte do material de aula apresenta uma demonstração de imputação múltipla para `WPA`. Ela não foi aplicada ao dataset persistido neste notebook porque o objetivo aqui é criar uma cópia tratada e confiável, sem substituir silenciosamente valores originais por estimativas. Se a disciplina exigir especificamente um exercício de imputação múltipla, ele deve ser apresentado em uma seção experimental separada, mantendo o valor original e documentando as hipóteses do modelo.